# Lab 02 — Gradient-Based and Blind Optimization

**Guide:** `practice/02_optimization.pdf` explains every task and the design decisions. Read it alongside.

Cells marked **TODO** are yours: replace `raise NotImplementedError` with your code. Every other cell is ready and
produces the plots once your functions work. Run the notebook from top to bottom.

| Part | Task | Time |
|:--|:--|:-:|
| A | Gradient descent: by hand and with JAX, on P1–P3 | 30 min |
| B | A genetic algorithm from scratch, on P1–P3 | 40 min |
| C | pyBlindOpt: initialization methods and stronger optimizers | 25 min |
| D | Newton's method with `jax.hessian` | 15 min |
| E | Challenge: a classifier trained on the 0/1 loss | home |

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pyBlindOpt as pbo
from lab02_utils import (
    PROBLEMS,
    Counted,
    Recorder,
    best_so_far,
    make_classification,
    plot_boxes,
    plot_convergence,
    plot_fit,
    plot_landscape,
    plot_paths,
    plot_population,
)
from pyBlindOpt import init, utils

plt.rcParams["figure.figsize"] = (9, 4)

## The three problems

| | loss $\mathcal{L}(\theta)$ | parameters $\theta$ | character |
|:--|:--|:--|:--|
| **P1** | $\frac1N\sum_i (w x_i + b - y_i)^2$ | $(w, b) \in [-5,5]^2$ | convex, elongated |
| **P2** | $(1-x)^2 + 100\,(y-x^2)^2$ | $(x, y) \in [-2,2]\times[-1,3]$ | curved valley |
| **P3** | $\frac1N\sum_i (\sin(\omega x_i + \varphi) - y_i)^2$ | $(\omega, \varphi) \in [0.1,4]\times[-\pi,\pi]$ | multimodal |

Each `PROBLEMS[key]` has `.loss` (JAX), `.numpy_loss`, `.bounds`, `.start`, `.optimum` and, for P1 and P3, `.data`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, problem in zip(axes, PROBLEMS.values()):
    plot_landscape(problem, ax)
    ax.plot(*problem.start, "wo", mec="k", ms=8)
    ax.set_title(f"{problem.name}\nwhite dot: GD start, red star: optimum", fontsize=10)
plt.tight_layout()
plt.show()

## Part A — Gradient descent

### A1. The gradient of P1 by hand

Derive $\partial\mathcal{L}/\partial w$ and $\partial\mathcal{L}/\partial b$ for P1 (guide, task A1) and implement
them. The next cell compares your result with `jax.grad`.

In [ ]:
# TODO: return the gradient of P1 at theta = (w, b) as an array of shape (2,).
data_p1 = PROBLEMS["P1"].data
assert data_p1 is not None
x_p1, y_p1 = data_p1


def grad_p1_manual(theta) -> np.ndarray:
    w, b = theta
    raise NotImplementedError("A1: gradient of the P1 loss")

In [ ]:
grad_p1_jax = jax.grad(PROBLEMS["P1"].loss)
for theta in [np.array([0.0, 0.0]), np.array([-4.0, 4.0]), np.array([1.5, -2.0])]:
    print(theta, "manual:", np.round(grad_p1_manual(theta), 6), " jax:", np.round(np.asarray(grad_p1_jax(theta)), 6))

### A2. Gradient descent

$\theta_{k+1} = \theta_k - \eta\,\nabla\mathcal{L}(\theta_k)$. Return the **whole path**: an array of shape
`(n_steps + 1, 2)` whose first row is `theta0`.

In [ ]:
# TODO: implement plain gradient descent.
def gradient_descent(grad, theta0, lr, n_steps) -> np.ndarray:
    raise NotImplementedError("A2: gradient descent")

### A3. Learning rates on P1, P2 and P3

Three learning rates per problem, from the problem's start point. Change them and answer the guide's questions.

In [ ]:
LEARNING_RATES = {"P1": [0.02, 0.2, 0.25], "P2": [2e-4, 1e-3, 2e-3], "P3": [0.01, 0.05, 0.2]}
N_STEPS = {"P1": 300, "P2": 3000, "P3": 1000}

gd_results = {}
for key, problem in PROBLEMS.items():
    grad = jax.jit(jax.grad(problem.loss))
    paths = {f"lr = {lr}": gradient_descent(grad, problem.start, lr, N_STEPS[key]) for lr in LEARNING_RATES[key]}
    finite = {k: p for k, p in paths.items() if np.all(np.isfinite(p)) and np.all(np.abs(p) < 1e3)}
    for label in paths.keys() - finite.keys():
        print(f"{key}, {label}: diverged")
    plot_paths(problem, finite, "- gradient descent")
    gd_results[key] = min(finite.values(), key=lambda p: problem.numpy_loss(p[-1]))[-1]
plot_fit(PROBLEMS["P1"], {"GD": gd_results["P1"]}, "- GD result")
plot_fit(PROBLEMS["P3"], {"GD": gd_results["P3"]}, "- GD result")

### A4. Multi-start on P3

Run gradient descent from `n_starts` random points of the box (uniform) and return the final points, shape
`(n_starts, 2)`.

In [ ]:
# TODO: multi-start gradient descent.
def multistart_gd(problem, n_starts, lr, n_steps, seed=0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    grad = jax.jit(jax.grad(problem.loss))
    raise NotImplementedError("A4: multi-start gradient descent")

In [ ]:
problem = PROBLEMS["P3"]
finals = multistart_gd(problem, n_starts=30, lr=0.05, n_steps=500)
hits = np.linalg.norm(finals - problem.optimum, axis=1) < 0.05
ax = plot_landscape(problem)
ax.scatter(finals[:, 0], finals[:, 1], c=np.where(hits, "tab:green", "tab:red"), edgecolor="k", zorder=3)
ax.set_title(f"P3: {hits.sum()} of {len(finals)} starts reach the global minimum ({30 * 500} gradient evaluations)")
plt.show()

## Part B — A genetic algorithm from scratch

Read the design considerations in the guide (Part B) first. Every function receives a NumPy random generator `rng`,
so a run is reproducible from its seed. `bounds` has shape `(d, 2)`: column 0 the lower, column 1 the upper limits.

In [ ]:
# TODO: the four GA operators.
def init_population(bounds, n_pop, rng) -> np.ndarray:
    """n_pop points drawn uniformly inside the box, shape (n_pop, d)."""
    raise NotImplementedError("B1: uniform initial population")


def tournament(pop, fitness, k, rng) -> np.ndarray:
    """Pick k individuals at random and return a copy of the best one (lowest fitness)."""
    raise NotImplementedError("B1: tournament selection")


def blend_crossover(p1, p2, alpha, rng) -> tuple[np.ndarray, np.ndarray]:
    """BLX-alpha: each gene of each child is uniform in [lo - alpha*d, hi + alpha*d], lo/hi the parents' genes."""
    raise NotImplementedError("B1: blend crossover")


def gaussian_mutation(child, bounds, rate, sigma_frac, rng) -> np.ndarray:
    """With probability `rate` per gene, add N(0, (sigma_frac * box width)^2); then clip to the box."""
    raise NotImplementedError("B1: Gaussian mutation")

### B1. The GA loop

1. Initialize and evaluate the population.
2. Each generation: keep the `n_elite` best as they are; fill the rest of the new population with children
   (two tournaments → crossover → mutation of each child); evaluate the new population.
3. Track the best individual ever seen.

Return `(best_theta, best_loss, history)` where `history` is the list of populations, one per generation
(including generation 0). Evaluate a whole population at once: `loss(pop)` returns one value per row.

In [ ]:
# TODO: the genetic algorithm.
def genetic_algorithm(
    loss, bounds, n_pop=30, n_gen=60, k=3, alpha=0.5, rate=0.5, sigma_frac=0.1, n_elite=2, seed=0
) -> tuple[np.ndarray, float, list[np.ndarray]]:
    rng = np.random.default_rng(seed)
    raise NotImplementedError("B1: the GA loop")

### B2. Run it on the three problems

Budget: 30 individuals × 61 generations = 1830 evaluations of the loss (and **no** gradient).

In [ ]:
ga_results = {}
for key, problem in PROBLEMS.items():
    theta, value, history = genetic_algorithm(problem.numpy_loss, problem.bounds, seed=1)
    ga_results[key] = theta
    print(f"{key}: GA best {np.round(theta, 4)}, loss {value:.5f}  |  GD best loss "
          f"{float(problem.numpy_loss(gd_results[key])):.5f}  |  optimum loss {float(problem.numpy_loss(problem.optimum)):.5f}")
    plot_population(problem, history, epochs=(0, 3, 10, -1), title="- your GA")
plot_fit(PROBLEMS["P3"], {"GD": gd_results["P3"], "GA": ga_results["P3"]}, "- GD vs. GA")

### B3. Selection pressure and mutation strength

10 seeds per setting on P3, with a **small budget** (15 generations) so the differences show. `k = 1` is random
selection (no pressure); `sigma_frac` sets the mutation step.

In [ ]:
settings = {
    "k=1": dict(k=1),
    "k=2": dict(k=2),
    "k=3 (default)": dict(k=3),
    "k=8": dict(k=8),
    "sigma=0.01": dict(sigma_frac=0.01),
    "sigma=0.3": dict(sigma_frac=0.3),
    "no elitism": dict(n_elite=0),
}
problem = PROBLEMS["P3"]
results = {name: [genetic_algorithm(problem.numpy_loss, problem.bounds, n_gen=15, seed=s, **kw)[1] for s in range(10)]
           for name, kw in settings.items()}
plot_boxes(results, title="P3: final loss over 10 seeds")

## Part C — pyBlindOpt: initialization and stronger optimizers

### C1. Initial populations

Return an initial population of `n_pop` points for `problem` using one of the methods below (see the guide and
notebook `02_blind_optimization.ipynb`, demo B2):

`"Random"`, `"LHS"`, `"Sobol"`, `"Chaotic"` (samplers in `pyBlindOpt.utils`), `"OBL"`, `"QOBL"`, `"OBLESA"`
(strategies in `pyBlindOpt.init`, which evaluate the loss to choose the points).

In [ ]:
INIT_METHODS = ["Random", "LHS", "Sobol", "Chaotic", "OBL", "QOBL", "OBLESA"]


# TODO: build the initial population.
def make_population(kind, problem, n_pop, rng) -> np.ndarray:
    raise NotImplementedError("C1: initial population")

In [ ]:
problem = PROBLEMS["P3"]
fig, axes = plt.subplots(1, len(INIT_METHODS), figsize=(3.2 * len(INIT_METHODS), 3.2))
for ax, kind in zip(axes, INIT_METHODS):
    pop = make_population(kind, problem, 20, np.random.default_rng(0))
    plot_landscape(problem, ax)
    ax.scatter(pop[:, 0], pop[:, 1], s=14, color="w", edgecolor="k")
    ax.set(title=f"{kind}\nbest {problem.numpy_loss(pop).min():.3f}", xlabel="", ylabel="")
plt.tight_layout()
plt.show()

### C2. Stronger optimizers

Run one pyBlindOpt optimizer by name — `"GA"`, `"DE"`, `"SHADE"` (DE with `variant="current-to-pbest/1/bin"`,
`policy="shade"`) or `"EGWO"` — from a given initial population, and return the best loss. Use `n_iter` epochs and
pass `seed` to the optimizer. Every optimizer then has the same budget: `len(population)` × (`n_iter` + 1).

In [ ]:
# TODO: run a pyBlindOpt optimizer.
def run_library(name, problem, population, n_iter, seed) -> float:
    raise NotImplementedError("C2: run a pyBlindOpt optimizer")

Budget per run: 20 individuals × 31 epochs = 620 evaluations, 15 seeds. First the optimizers (random
initialization, plus your GA with the same budget), then the initialization methods (with DE).

In [ ]:
N_POP, N_ITER, SEEDS = 20, 30, range(15)
for key, problem in PROBLEMS.items():
    optimum = float(problem.numpy_loss(problem.optimum))
    res = {name: [run_library(name, problem, make_population("Random", problem, N_POP, np.random.default_rng(s)),
                              N_ITER, s) - optimum for s in SEEDS]
           for name in ["GA", "DE", "SHADE", "EGWO"]}
    res["your GA"] = [genetic_algorithm(problem.numpy_loss, problem.bounds, n_pop=N_POP, n_gen=N_ITER, seed=s)[1]
                      - optimum for s in SEEDS]
    plot_boxes(res, title=f"{problem.name}: optimizers (620 evaluations, 15 seeds)", ylabel="final loss - optimum")

In [ ]:
for key, problem in PROBLEMS.items():
    optimum = float(problem.numpy_loss(problem.optimum))
    res = {kind: [run_library("DE", problem, make_population(kind, problem, N_POP, np.random.default_rng(s)),
                              N_ITER, s) - optimum for s in SEEDS]
           for kind in INIT_METHODS}
    plot_boxes(res, title=f"{problem.name}: DE with each initialization (15 seeds)", ylabel="final loss - optimum")

### C3. Watch one run

The class interface keeps the optimizer object, so a `Recorder` callback can store every population.

In [ ]:
problem = PROBLEMS["P3"]
opt = pbo.DifferentialEvolution(problem.numpy_loss, problem.bounds, n_pop=20, n_iter=30, seed=3)
rec = Recorder.attach(opt)
opt.optimize()
plot_population(problem, rec.pops, epochs=(0, 3, 10, -1), title="- pyBlindOpt DE")
plot_convergence({"DE": best_so_far(rec.pops, problem)}, title="P3: best loss so far")

## Part D — Newton's method

$\theta_{k+1} = \theta_k - (H + \lambda I)^{-1}\,\nabla\mathcal{L}(\theta_k)$, with $H$ from `jax.hessian`.
$\lambda = 0$ is pure Newton; $\lambda > 0$ ("damping") blends it with a small gradient step. Return the path.

In [ ]:
# TODO: (damped) Newton's method.
def newton(problem, theta0, n_steps, damping=0.0) -> np.ndarray:
    grad = jax.jit(jax.grad(problem.loss))
    hess = jax.jit(jax.hessian(problem.loss))
    raise NotImplementedError("D1: Newton's method")

In [ ]:
for key, problem in PROBLEMS.items():
    grad = jax.jit(jax.grad(problem.loss))
    lr = LEARNING_RATES[key][1]
    paths = {
        f"GD lr={lr}, 20 steps": gradient_descent(grad, problem.start, lr, 20),
        "Newton, 20 steps": newton(problem, problem.start, 20),
        "damped Newton (lambda=1), 20 steps": newton(problem, problem.start, 20, damping=1.0),
    }
    for label, p in paths.items():
        print(f"{key} {label:36s} final theta {np.round(p[-1], 4)}, loss {float(problem.numpy_loss(p[-1])):.5f}")
    plot_paths(problem, paths, "- GD vs. Newton")

## Part E — Challenge: optimizing what we actually care about

A linear classifier predicts $\hat y = [\,w^\top x + b > 0\,]$. We want the lowest **error rate** (0/1 loss), but
its gradient is zero almost everywhere. Two routes (guide, Part E):

1. **Surrogate + gradient:** minimize the smooth logistic loss with gradient descent (what logistic regression does).
2. **Blind:** minimize the error rate itself with a population method.

$\theta = (w_1, \dots, w_6, b) \in [-5, 5]^7$.

In [ ]:
X_train, y_train, X_test, y_test = make_classification(seed=0)
D = X_train.shape[1] + 1
BOUNDS_E = np.array([[-5.0, 5.0]] * D)
print("train", X_train.shape, "test", X_test.shape, "class balance", y_train.mean().round(2))


# TODO: error rate of theta on (X, y). theta has shape (D,) or (n_pop, D); return a float or an array (n_pop,).
def error_rate(theta, X, y) -> float | np.ndarray:
    raise NotImplementedError("E1: 0/1 loss")


# TODO: mean logistic loss (binary cross-entropy) of theta on (X, y), written with jax.numpy.
def logistic_loss(theta, X, y) -> jax.Array:
    raise NotImplementedError("E2: logistic loss")

In [ ]:
theta0 = np.zeros(D)
zero_one = lambda t: jnp.mean((X_train @ t[:-1] + t[-1] > 0) != y_train)  # noqa: E731
print("gradient of the 0/1 loss at a random point:", jax.grad(lambda t: zero_one(t).astype(float))(np.ones(D)))
print("gradient of the logistic loss at 0:       ", np.round(np.asarray(jax.grad(logistic_loss)(theta0, X_train, y_train)), 4))

In [ ]:
SEEDS_E = range(10)
grad_log = jax.jit(jax.grad(logistic_loss))
theta_log = gradient_descent(lambda t: grad_log(t, X_train, y_train), theta0, lr=0.1, n_steps=2000)[-1]
results_e = {"GD on logistic loss": [float(error_rate(theta_log, X_test, y_test))] * len(SEEDS_E)}


def train_error(theta):
    return error_rate(theta, X_train, y_train)


for name, fn, kw in [
    ("DE on 0/1 loss", pbo.differential_evolution, {}),
    ("SHADE on 0/1 loss", pbo.differential_evolution, dict(variant="current-to-pbest/1/bin", policy="shade")),
    ("EGWO on 0/1 loss", pbo.enhanced_grey_wolf_optimization, {}),
]:
    results_e[name] = []
    for s in SEEDS_E:
        counted = Counted(train_error)
        theta, _ = fn(counted, BOUNDS_E, n_pop=40, n_iter=150, seed=s, **kw)
        results_e[name].append(float(error_rate(theta, X_test, y_test)))
results_e["your GA on 0/1 loss"] = [
    float(error_rate(genetic_algorithm(train_error, BOUNDS_E, n_pop=40, n_gen=150, seed=s)[0], X_test, y_test))
    for s in SEEDS_E
]
print(f"evaluations per blind run: {counted.n}")
for name, errs in results_e.items():
    print(f"{name:24s} test error: median {np.median(errs):.3f}, min {np.min(errs):.3f}, max {np.max(errs):.3f}")
plot_boxes(results_e, title="Test error over 10 seeds (label noise 10%)", ylabel="test error", log=False)